<link rel="stylesheet" href="notebooks/styles.css">

<div class="title-wrap">
  <h1 class="title-main" style="font-weight: bold; font-size: 2.65rem; margin-bottom: 0.5rem;">
  Spatial Data Science Approaches to Wildfire Severity Modeling
</h1>
<h2 class="title-sub" style="font-style: italic; font-size: 1.8rem; margin-top: 0rem; margin-bottom: 0.2rem;">
  A GIS‑Driven, Tree‑Based Machine Learning Analysis of California Wildfires
</h2>
</div>

# Module 10B: *Fire Damage Feature Ablation*

## Metadata
---
### Contents  
> 1. **
> 2. **
> 3. **
---
### Notes
---
### Inputs
---
### Outputs 
---
### User Defined Dependencies

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

from src.data_utils import *
from src.model_utils import *
from src.plot_utils import *

---
### Third Party Dependencies

In [2]:
# Core libraries
import pandas as pd
import numpy as np
import json

# Modeling libraries
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.model_selection import train_test_split

---

##  Load Files

In [3]:
X_damage = pd.read_csv('../data/processed/X_damage.csv')
y_damage = pd.read_csv('../data/processed/y_damage.csv').squeeze()  # Load as Series
details_damage = pd.read_csv('../data/processed/details_damage.csv')

pal_details = pd.read_csv('../data/processed/pal_details.csv')
pal_X = pd.read_csv('../data/processed/pal_X.csv')
pal_y = pd.read_csv('../data/processed/pal_y.csv')

best_strategy = pd.read_csv('../data/processed/damage_best_strategy.csv')

with open('model_parameters_ignition.json', 'r') as f:
    model_parameters = json.load(f)

with open('feature_sets.json', 'r') as f:
    feature_sets = json.load(f)

## Subset

In [4]:
reform = pd.concat([X_damage,y_damage], axis=1)
subset = subset_df(reform, 'Target_Damage', 500)

y = subset['Target_Damage']
X = subset.drop(columns='Target_Damage')

In [5]:
X_rf, y_rf = apply_balancing('RF', best_strategy, X, y)
X_xgb, y_xgb = apply_balancing('XGB', best_strategy, X, y)

## Build Models

In [6]:
RF_parameters = model_parameters['Random Forest']
XGB_parameters = model_parameters['XGBoost']

# Build Final tuned models
damage_xgb = xgb.XGBClassifier(**XGB_parameters)
damage_rf = RandomForestClassifier(**RF_parameters)

display(RF_parameters)
display(XGB_parameters)

{'n_estimators': 150,
 'max_depth': 15,
 'min_samples_split': 2,
 'max_features': 'log2',
 'class_weight': 'balanced'}

{'objective': 'multi:softmax',
 'num_class': 3,
 'n_estimators': 100,
 'max_depth': 4,
 'learning_rate': 0.1,
 'verbosity': 0}

In [7]:
models = {
    "XGB": damage_xgb, 
    "RF": damage_rf
}

## SHAP

In [8]:
xgb_top_10 = get_shap(damage_xgb, X_xgb, y_xgb)
xgb_top_10


,0,1
0,road_density_x_forest_percent,0.363505
1,Solar Radiation 7 Day Avg,0.339217
2,1000-hour Dead Fuel Moisture,0.279197
3,Season_Summer,0.228033
4,Minimum Relative Humidity 7 Day Avg,0.107813
5,wetlands_percent,0.101964
6,elevation_range,0.096776
7,Daily Minimum Air Temperature,0.092409
8,Vapor Pressure Deficit,0.087097
9,Santa_Ana_Score,0.082662


In [9]:
rf_top_10 = get_shap(damage_rf, X_rf, y_rf)
rf_top_10 


,0,1
0,1000-hour Dead Fuel Moisture,0.023708
1,road_density_x_forest_percent,0.020365
2,forest_percent,0.018180
3,Season_Summer,0.016567
4,Vapor Pressure Deficit_x_Solar Radiation,0.016175
5,Daily Maximum Air Temperature 7 Day Avg,0.016072
6,Solar Radiation 7 Day Avg,0.015037
7,Solar Radiation,0.015029
8,Vapor Pressure Deficit,0.013651
9,Minimum Relative Humidity 7 Day Avg,0.013240


## Ablation

In [10]:
for set_name, columns in feature_sets.items():
    print(f"{set_name}: {columns}")
    print("\n")

Water Demand: ['Actual Evapotranspiration', 'Solar Radiation', 'Solar Radiation 7 Day Avg', 'Daily Minimum Air Temperature', 'Daily Maximum Air Temperature', 'Daily Maximum Air Temperature 7 Day Avg', 'Daily Minimum Air Temperature 7 Day Avg', 'Vapor Pressure Deficit', 'Vapor Pressure Deficit 7 Day Avg', 'Wind Speed', 'Wind Speed 7 Day Avg']


Water Supply: ['Precipitation', 'Precipitation 7 Day Avg', 'Maximum Relative Humidity', 'Minimum Relative Humidity', 'Maximum Relative Humidity 7 Day Avg', 'Minimum Relative Humidity 7 Day Avg', 'Specific Humidity', '100-hour Dead Fuel Moisture', '1000-hour Dead Fuel Moisture']


Water Supply Indexes: ['Standardized Precipitation Index 30-Day', 'Standardized Precipitation Index 180-Day', 'Standardized Precipitation Evapotranspiration Index 30-Day', 'Standardized Precipitation Evapotranspiration Index 90-Day', 'Standardized Precipitation Evapotranspiration Index 180-Day', 'Palmer Drought Severity Index']


Fire Danger: ['Burning Index', 'Energy Re

In [11]:
compare = []

for set_name, set_list in feature_sets.items():
    for model_name, model in models.items():
        print("Testing " + f"{model_name}: {set_name}")
        X_section = X.drop(columns = set_list)
        compare.append(compare_model(model, X_section, y, best_strategy,
                                     name = model_name, test_set = set_name))

Testing XGB: Water Demand
Testing RF: Water Demand
Testing XGB: Water Supply
Testing RF: Water Supply
Testing XGB: Water Supply Indexes
Testing RF: Water Supply Indexes
Testing XGB: Fire Danger
Testing RF: Fire Danger
Testing XGB: Social
Testing RF: Social
Testing XGB: Temporal
Testing RF: Temporal
Testing XGB: Elevation
Testing RF: Elevation
Testing XGB: WUI
Testing RF: WUI
Testing XGB: Ecoregion
Testing RF: Ecoregion
Testing XGB: Land Cover
Testing RF: Land Cover
Testing XGB: Interactions
Testing RF: Interactions
Testing XGB: Wind Slope
Testing RF: Wind Slope
Testing XGB: Others
Testing RF: Others


In [12]:
comparisons = pd.DataFrame(compare)
comparisons

,Test Set,Model,Weighted F1,Macro F1,High Risk Recall
0,Water Demand,XGB,0.809808,0.808081,0.834862
1,Water Demand,RF,0.784897,0.783042,0.807339
2,Water Supply,XGB,0.790169,0.788647,0.798165
3,Water Supply,RF,0.800000,0.798367,0.816514
4,Water Supply Indexes,XGB,0.790000,0.788285,0.807339
5,Water Supply Indexes,RF,0.794674,0.792715,0.825688
6,Fire Danger,XGB,0.795088,0.793508,0.807339
7,Fire Danger,RF,0.789788,0.787879,0.816514
8,Social,XGB,0.774892,0.772951,0.798165
9,Social,RF,0.784897,0.783042,0.807339
